In [1]:
import os
import requests, json, re, pprint

# === CONFIGURATION ===
API_KEY = os.environ.get("LLM_API_KEY", "your-api-key-here")
API_HOST = os.environ.get("LLM_HOST", "localhost")
API_PORT = int(os.environ.get("LLM_PORT", "50001"))

# === STUDENT INPUT ===
student_background = {
    "department": "Computer Science",
    "degree_level": "Undergraduate",
    "completed_courses": [
        "CSCI 256 Programming in Python",
        "CSCI 356 Data Structures in Python",
        "CSCI 343 Fundamentals of Data Science"
    ]
}
target_course = "Deep Learning"

# === PROMPT BUILDER ===
def build_prompt(student_background, target_course):
    background = (
        f"Department: {student_background['department']}\n"
        f"Level: {student_background['degree_level']}\n"
        f"Completed Courses: {', '.join(student_background['completed_courses'])}\n"
    )

    course_list = """
CSCI 256 Programming in Python
CSCI 343 Fundamentals of Data Science
CSCI 356 Data Structures in Python
CSCI 433 Algorithm and Data Structure Analysis
CSCI 443 Advanced Data Science
CSCI 475 Introduction to Database Systems
CSCI 345 Information Storage and Retrieval
CSCI 444 Information Visualization
CSCI 517 Natural Language Processing
CSCI 543 Data Mining
CSCI 632 Machine Learning
CSCI 581 Special Topics in Computer Science (Computer Vision)
CSCI 492 Special Topics in Data Science (Deep Learning - Undergraduate)
ENGR 691 Special Topics in Engineering Science (Deep Learning - Graduate)
"""

    prompt = f"""
You are an academic advisor. Based on the student profile and available courses,
return only a valid JSON object. Do not add any explanation, notes, or markdown.

Student Profile:
{background}
Target Course: {target_course}

Available Courses:
{course_list}

Return a JSON like this (no explanation):

{{
  "target_course": "CSCI 492 Special Topics in Data Science (Deep Learning - Undergraduate)",
  "recommended_prerequisites": [
    {{ "course_number": "CSCI 356", "course_name": "Data Structures in Python" }},
    {{ "course_number": "CSCI 343", "course_name": "Fundamentals of Data Science" }}
  ],
  "suggested_learning_path": [
    {{ "step": 1, "course_number": "CSCI 343", "course_name": "Fundamentals of Data Science" }},
    {{ "step": 2, "course_number": "CSCI 632", "course_name": "Machine Learning" }},
    {{ "step": 3, "course_number": "CSCI 492", "course_name": "Special Topics in Data Science (Deep Learning - Undergraduate)" }}
  ]
}}
""".strip()
    return prompt

# === LLM API CALL ===
def query_local_llm(prompt):
    url = f"http://{API_HOST}:{API_PORT}/generate"
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    payload = {
        "text": prompt,
        "max_tokens": 2048,
        "temperature": 0.1
    }
    response = requests.post(url, json=payload, headers=headers)
    result = response.json()
    if "text" not in result:
        raise ValueError(f"❌ LLM Error: {result.get('error', {}).get('message', 'Unknown error')}")
    return result["text"]

# === JSON PARSER ===
def extract_learning_plan(raw_text):
    all_blocks = re.findall(r'\{.*?\}', raw_text, re.DOTALL)
    for block in all_blocks:
        try:
            return json.loads(block)
        except json.JSONDecodeError:
            continue
    print("❌ No valid JSON object found in response.")
    print("LLM output was:\n", raw_text)
    raise ValueError("No valid JSON found.")

# === RETRY + EXECUTION ===
def get_structured_plan_with_retry(prompt, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            response_text = query_local_llm(prompt)
            print(f"\n🧾 Raw response from LLM (attempt {attempt + 1}):\n{response_text}\n")
            return extract_learning_plan(response_text)
        except Exception as e:
            print(f"\n⚠️ Attempt {attempt + 1} failed: {e}\nRetrying...\n")
    raise RuntimeError("❌ All retry attempts failed.")

# === RUN ===
prompt = build_prompt(student_background, target_course)
print("---- Prompt Sent ----\n", prompt[:500], "\n... (truncated)\n")
structured_plan = get_structured_plan_with_retry(prompt)
print("\n✅ Final Structured Plan:")
pprint.pprint(structured_plan)


---- Prompt Sent ----
 You are an academic advisor. Based on the student profile and available courses,
return only a valid JSON object. Do not add any explanation, notes, or markdown.

Student Profile:
Department: Computer Science
Level: Undergraduate
Completed Courses: CSCI 256 Programming in Python, CSCI 356 Data Structures in Python, CSCI 343 Fundamentals of Data Science

Target Course: Deep Learning

Available Courses:

CSCI 256 Programming in Python
CSCI 343 Fundamentals of Data Science
CSCI 356 Data Structures  
... (truncated)


🧾 Raw response from LLM (attempt 1):
 {
  "target_course": "CSCI 492 Special Topics in Data Science (Deep Learning - Undergraduate)",
  "recommended_prerequisites": [
    { "course_number": "CSCI 356", "course_name": "Data Structures in Python" },
    { "course_number": "CSCI 343", "course_name": "Fundamentals of Data Science" }
  ],
  "suggested_learning_path": [
    { "step": 1, "course_number": "CSCI 343", "course_name": "Fundamentals of Data Science

In [4]:
import os
#part2
import requests, json, re, pprint

# === CONFIGURATION ===
API_KEY = os.environ.get("LLM_API_KEY", "your-api-key-here")
API_HOST = os.environ.get("LLM_HOST", "localhost")
API_PORT = int(os.environ.get("LLM_PORT", "50001"))

# === STUDENT INPUT ===
student_background = {
    "department": "Computer Science",
    "degree_level": "Undergraduate",
    "completed_courses": [
        "CSCI 256 Programming in Python",
        "CSCI 356 Data Structures in Python",
        "CSCI 343 Fundamentals of Data Science"
    ]
}
target_course = "Deep Learning"

# === PROMPT BUILDER WITH EXAMPLES & JSON SEEDING ===
def build_prompt_with_examples_and_seed(student_background, target_course):
    examples = """
Example Student Plans:

Student A:
Background: Graduate student (non-CS undergrad), took Python and stats.
Plan:
{
  "target_course": "Deep Learning",
  "recommended_prerequisites": [
    { "course_number": "CSCI 543", "course_name": "Data Mining" },
    { "course_number": "CSCI 632", "course_name": "Machine Learning" }
  ],
  "suggested_learning_path": [
    { "step": 1, "course_number": "CSCI 543", "course_name": "Data Mining" },
    { "step": 2, "course_number": "CSCI 632", "course_name": "Machine Learning" },
    { "step": 3, "course_number": "CSCI 492", "course_name": "Deep Learning - Undergraduate" }
  ]
}

Student B:
Background: Graduate in Mechanical Engg, took Python and ML projects.
Plan:
{
  "target_course": "Deep Learning",
  "recommended_prerequisites": [],
  "suggested_learning_path": [
    { "step": 1, "course_number": "CSCI 492", "course_name": "Deep Learning - Undergraduate" }
  ]
}
"""

    current_student = f'''
{examples}

Student C:
Background: {student_background['degree_level']} in {student_background['department']}, took {', '.join(student_background['completed_courses'])}.
Plan:
{{
  "target_course": "Deep Learning",
'''
    return current_student.strip()

# === LLM CALL ===
def query_local_llm(prompt):
    url = f"http://{API_HOST}:{API_PORT}/generate"
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    payload = {
        "text": prompt,
        "max_tokens": 1024,
        "temperature": 0.1
    }
    response = requests.post(url, json=payload, headers=headers)
    result = response.json()
    if "text" not in result:
        raise ValueError(f"❌ LLM Error: {result.get('error', {}).get('message', 'Unknown error')}")
    return result["text"]

# === JSON EXTRACTOR & REPAIR ===
def extract_learning_plan_with_seeded_prefix(generated_text):
    full_json = '''{
  "target_course": "Deep Learning",
''' + generated_text.strip()

    # Cut off any trailing "Student D", etc.
    cutoff = re.search(r'\n(Student|Example)\s+\w+:', full_json)
    if cutoff:
        full_json = full_json[:cutoff.start()]

    # Trim to last closing brace
    last_valid = full_json.rfind("}")
    if last_valid != -1:
        full_json = full_json[:last_valid + 1]

    # Balance brackets
    if full_json.count("{") > full_json.count("}"):
        full_json += "}"
    if full_json.count("[") > full_json.count("]"):
        full_json += "]"

    # Remove trailing commas before closing brackets
    full_json = re.sub(r',\s*([\]}])', r'\1', full_json)

    # Fix unquoted course numbers
    full_json = re.sub(r'"course_number":\s*([A-Z]+\s*\d+)', r'"course_number": "\1"', full_json)

    try:
        return json.loads(full_json)
    except json.JSONDecodeError as e:
        print("❌ Final JSON still malformed:", e)
        print("Sanitized JSON string:\n", full_json)
        raise

# === RETRY EXECUTION ===
def get_structured_plan_with_retry(prompt_fn, max_attempts=3):
    prompt = prompt_fn(student_background, target_course)
    for attempt in range(max_attempts):
        try:
            response = query_local_llm(prompt)
            print(f"\n🧾 Raw response (Attempt {attempt + 1}):\n{response}\n")
            return extract_learning_plan_with_seeded_prefix(response)
        except Exception as e:
            print(f"⚠️ Attempt {attempt + 1} failed:", e, "\nRetrying...\n")
    raise RuntimeError("❌ All retry attempts failed.")

# === RUN ===
structured_plan = get_structured_plan_with_retry(build_prompt_with_examples_and_seed)
print("\n✅ Final Structured Plan:")
pprint.pprint(structured_plan)



🧾 Raw response (Attempt 1):
 // because the next step after undergrad courses leading to ML is Probably CSCI 632 and then CSCI 492 for Deep Learning?
  "recommended_prerequisites": [],
  "suggested_learning_path": [
    { "step": 1, "course_number": "CSCI 632", "course_name": "Machine Learning" },
    { "step": 2, "course_number": "CSCI 492", "course_name": "Deep Learning - Undergraduate" }
  ]
}

Student D:
Background: Has taken CSCI 632 (ML) and may

❌ Final JSON still malformed: Expecting property name enclosed in double quotes: line 3 column 1 (char 38)
Sanitized JSON string:
 {
  "target_course": "Deep Learning",
// because the next step after undergrad courses leading to ML is Probably CSCI 632 and then CSCI 492 for Deep Learning?
  "recommended_prerequisites": [],
  "suggested_learning_path": [
    { "step": 1, "course_number": "CSCI 632", "course_name": "Machine Learning" },
    { "step": 2, "course_number": "CSCI 492", "course_name": "Deep Learning - Undergraduate" }
  ]
}
⚠️

In [11]:
import os
#part3(a)
import requests, pprint

# === CONFIGURATION ===
API_KEY = os.environ.get("LLM_API_KEY", "your-api-key-here")
API_HOST = os.environ.get("LLM_HOST", "localhost")
API_PORT = int(os.environ.get("LLM_PORT", "50001"))

# === STUDENT INPUT ===
student_background = {
    "department": "Computer Science",
    "degree_level": "Undergraduate",
    "completed_courses": [
        "CSCI 256 Programming in Python",
        "CSCI 343 Fundamentals of Data Science"
    ]
}
target_course = "Deep Learning"

# === BUILD PROMPT WITH COT FORMAT ===
def build_cot_prompt(student_background, target_course):
    return f"""
You are an academic advisor. A student wants to take the course: "{target_course}".

Their profile:
- Degree Level: {student_background['degree_level']}
- Department: {student_background['department']}
- Completed Courses: {', '.join(student_background['completed_courses'])}

Your task:
1. Identify foundational concepts required for "{target_course}" (e.g., ML, linear algebra, calculus).
2. Compare these with the student's background and identify missing topics.
3. ONLY recommend courses from the following catalog to address gaps:

- CSCI 256: Programming in Python
- CSCI 343: Fundamentals of Data Science
- CSCI 356: Data Structures in Python
- CSCI 433: Algorithm and Data Structure Analysis
- CSCI 443: Advanced Data Science
- CSCI 475: Introduction to Database Systems
- CSCI 345: Information Storage and Retrieval
- CSCI 444: Information Visualization
- CSCI 517: Natural Language Processing
- CSCI 543: Data Mining
- CSCI 632: Machine Learning
- CSCI 581: Computer Vision
- CSCI 492: Deep Learning (Undergraduate)
- ENGR 691: Deep Learning (Graduate)

Be concise and focused. Do not include unrelated or hallucinated courses.

---

Now reason step-by-step.

At the end, output the plan as:

Final Recommendation (Structured Plan):
Step 1: [Course Number and Name] — [Justification]
Step 2: ...
Step N: CSCI 492 — Final Target Course
""".strip()

# === CALL LOCAL LLM ===
def query_local_llm(prompt):
    url = f"http://{API_HOST}:{API_PORT}/generate"
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    payload = {
        "text": prompt,
        "max_tokens": 1000,
        "temperature": 0.3
    }
    response = requests.post(url, json=payload, headers=headers)
    result = response.json()
    if "text" not in result:
        raise ValueError(f"❌ LLM Error: {result.get('error', {}).get('message', 'Unknown error')}")
    return result["text"]

# === EXECUTE WITH RETRY ===
def run_cot_reasoning(prompt_fn, max_attempts=3):
    prompt = prompt_fn(student_background, target_course)
    print("📤 Prompt Preview:\n", prompt[:600], "\n... (truncated)\n")
    for attempt in range(max_attempts):
        try:
            response = query_local_llm(prompt)
            print(f"\n🧾 Raw response (Attempt {attempt + 1}):\n\n{response}\n")
            return response
        except Exception as e:
            print(f"⚠️ Attempt {attempt + 1} failed:", e, "\nRetrying...\n")
    raise RuntimeError("❌ All retry attempts failed.")

# === RUN ===
cot_output = run_cot_reasoning(build_cot_prompt)


📤 Prompt Preview:
 You are an academic advisor. A student wants to take the course: "Deep Learning".

Their profile:
- Degree Level: Undergraduate
- Department: Computer Science
- Completed Courses: CSCI 256 Programming in Python, CSCI 343 Fundamentals of Data Science

Your task:
1. Identify foundational concepts required for "Deep Learning" (e.g., ML, linear algebra, calculus).
2. Compare these with the student's background and identify missing topics.
3. ONLY recommend courses from the following catalog to address gaps:

- CSCI 256: Programming in Python
- CSCI 343: Fundamentals of Data Science
- CSCI 356: Dat 
... (truncated)


🧾 Raw response (Attempt 1):

 **Final Recommendation (Structured Plan):**  
Step 1: **CSCI 632: Machine Learning** — Prerequisite foundational course for deep learning. The student has taken CSCI 343 (Data Science) but lacks formal machine learning training, which covers core concepts like supervised/unsupervised learning, classification, regression, and opti

In [18]:
import os
#part3(b)
import requests, pprint

# === CONFIGURATION ===
API_KEY = os.environ.get("LLM_API_KEY", "your-api-key-here")
API_HOST = os.environ.get("LLM_HOST", "localhost")
API_PORT = int(os.environ.get("LLM_PORT", "50001"))

# === STUDENT INPUT ===
student_background = {
    "degree_level": "Undergraduate",
    "department": "Computer Science",
    "completed_courses": [
        "CSCI 256 Programming in Python",
        "CSCI 343 Fundamentals of Data Science"
    ]
}
target_course = "Deep Learning"

# === FIXED STEP 1: Required Knowledge Topics (Static)
required_knowledge = """
- Programming in Python
- Machine Learning
- Optimization Algorithms (e.g., Gradient Descent)
- Data Handling and Preprocessing
- Neural Networks (basic architecture, backpropagation)

(Note: Probability, Statistics, Linear Algebra, and Multivariable Calculus are assumed to be covered in the student's general CS curriculum and do not need to be checked.)
"""

# === LLM CALL ===
def query_local_llm(prompt, temperature=0.3):
    url = f"http://{API_HOST}:{API_PORT}/generate"
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    payload = {
        "text": prompt,
        "max_tokens": 1000,
        "temperature": temperature
    }
    response = requests.post(url, json=payload, headers=headers)
    result = response.json()
    if "text" not in result:
        raise ValueError(f"❌ LLM Error: {result.get('error', {}).get('message', 'Unknown error')}")
    return result["text"].strip()

# === PROMPT 2: Compare Required Knowledge with Student Background
def prompt_compare_with_student(required_knowledge, background):
    return f"""
You are an academic advisor.

Here is the required knowledge for the course "CSCI 492: Deep Learning":

{required_knowledge}

Student Profile:
- Degree Level: {background['degree_level']}
- Department: {background['department']}
- Completed Courses: {', '.join(background['completed_courses'])}

Compare the two and output:

Covered:
- Topic A
- Topic B

Missing:
- Topic C
- Topic D
""".strip()

# === PROMPT 3: Recommend Plan Based on Gaps (fully constrained version)
def prompt_generate_plan(missing_topics):
    return f"""
The student is missing the following required topics to take "CSCI 492: Deep Learning":

{missing_topics}

Your task is to return the minimum number of prerequisite courses from the catalog below that will fill the gaps in the correct order.

⚠️ DO NOT explain, reason, comment, or add jokes.
⚠️ DO NOT repeat previous steps or user instructions.
⚠️ ONLY output the plan using the exact format shown.

Course Catalog:
- CSCI 256: Programming in Python
- CSCI 343: Fundamentals of Data Science
- CSCI 356: Data Structures in Python
- CSCI 433: Algorithm and Data Structure Analysis
- CSCI 443: Advanced Data Science
- CSCI 475: Introduction to Database Systems
- CSCI 345: Information Storage and Retrieval
- CSCI 444: Information Visualization
- CSCI 517: Natural Language Processing
- CSCI 543: Data Mining
- CSCI 632: Machine Learning
- CSCI 581: Computer Vision
- CSCI 492: Deep Learning (Undergraduate)

📋 Output Format (Start directly below):

Step 1: [Course Number] — [Why it’s needed]
Step 2: ...
Final Step: CSCI 492 — Deep Learning (Target Course)

Step 1:
""".strip()

# === CHAIN EXECUTION ===
def run_prompt_chaining(student_background, target_course):
    print("🔍 Step 1 — Required Knowledge:\n", required_knowledge)

    # Step 2: Compare with background
    prompt2 = prompt_compare_with_student(required_knowledge, student_background)
    missing_topics = query_local_llm(prompt2)
    print("\n📊 Step 2 — Knowledge Comparison:\n", missing_topics)

    # Step 3: Generate learning plan
    prompt3 = prompt_generate_plan(missing_topics)
    final_plan = query_local_llm(prompt3)
    print("\n📚 Step 3 — Final Learning Plan:\n", final_plan)

# === RUN ALL STEPS ===
run_prompt_chaining(student_background, target_course)


🔍 Step 1 — Required Knowledge:
 
- Programming in Python
- Machine Learning
- Optimization Algorithms (e.g., Gradient Descent)
- Data Handling and Preprocessing
- Neural Networks (basic architecture, backpropagation)

(Note: Probability, Statistics, Linear Algebra, and Multivariable Calculus are assumed to be covered in the student's general CS curriculum and do not need to be checked.)


📊 Step 2 — Knowledge Comparison:
 Okay, let's see. The student is an undergrad in Computer Science, and they've taken CSCI 256 Programming in Python and CSCI 343 Fundamentals of Data Science. The required knowledge for the Deep Learning course includes Programming in Python, Machine Learning, Optimization Algorithms (like Gradient Descent), Data Handling and Preprocessing, and Neural Networks basics.

First, check the Programming in Python part. They have CSCI 256, which is a programming course in Python, so that's covered. 

Next, Machine Learning: The student's completed courses don't list any ML-sp